In [1]:
# All imports we need for this project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import os
print(os.getcwd())

C:\Users\Harleen\rossmann-forecasting


In [4]:
# Load all three files
train = pd.read_csv('data/train.csv',
                    dtype={'StateHoliday': str},
                    low_memory=False)

store = pd.read_csv('data/store.csv')

test  = pd.read_csv('data/test.csv',
                    dtype={'StateHoliday': str},
                    low_memory=False)

print("Train shape:", train.shape)
print("Store shape:", store.shape)
print("Test shape :", test.shape)

Train shape: (1017209, 9)
Store shape: (1115, 10)
Test shape : (41088, 8)


In [6]:
# See first 5 rows of train
print("=== TRAIN ===")
display(train.head())

print("=== STORE ===")
display(store.head())

=== TRAIN ===


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


=== STORE ===


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.00,9.00,2008.00,0,NaN,NaN,NaN
1,2,a,a,570.00,11.00,2007.00,1,13.00,2010.00,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.00,12.00,2006.00,1,14.00,2011.00,"Jan,Apr,Jul,Oct"
3,4,c,c,620.00,9.00,2009.00,0,NaN,NaN,NaN
4,5,a,a,29910.00,4.00,2015.00,0,NaN,NaN,NaN


In [7]:
# Find all missing values
print("Missing in train:")
print(train.isnull().sum())

print("\nMissing in store:")
print(store.isnull().sum())

Missing in train:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

Missing in store:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [8]:
# StateHoliday has mixed '0' and 0 — fix it
train['StateHoliday'] = train['StateHoliday'].astype(str)
train['StateHoliday'] = train['StateHoliday'].replace('0', 'none')
train['StateHoliday'] = train['StateHoliday'].replace({'a': 'public',
                                                        'b': 'easter',
                                                        'c': 'christmas'})

test['StateHoliday'] = test['StateHoliday'].astype(str)
test['StateHoliday'] = test['StateHoliday'].replace('0', 'none')
test['StateHoliday'] = test['StateHoliday'].replace({'a': 'public',
                                                      'b': 'easter',
                                                      'c': 'christmas'})

print("StateHoliday values:", train['StateHoliday'].unique())

StateHoliday values: ['none' 'public' 'easter' 'christmas']


In [9]:
# Fill missing competition distance with median
store['CompetitionDistance'].fillna(
    store['CompetitionDistance'].median(), inplace=True)

# Fill missing competition open date with 0
store['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
store['CompetitionOpenSinceYear'].fillna(0, inplace=True)

# Fill missing promo2 info with 0 / 'None'
store['Promo2SinceWeek'].fillna(0, inplace=True)
store['Promo2SinceYear'].fillna(0, inplace=True)
store['PromoInterval'].fillna('None', inplace=True)

print("Missing in store after fix:")
print(store.isnull().sum())

Missing in store after fix:
Store                        0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64


In [10]:
# Merge store info into train and test
train = train.merge(store, on='Store', how='left')
test  = test.merge(store, on='Store', how='left')

# Remove closed store rows from training data
train = train[train['Open'] == 1].copy()

# Remove rows with 0 sales
train = train[train['Sales'] > 0].copy()

print("Train shape after cleaning:", train.shape)
print("Test shape after merge    :", test.shape)

Train shape after cleaning: (844338, 18)
Test shape after merge    : (41088, 17)


In [11]:
# Convert Date from string to proper datetime
train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print("Train date range:", train['Date'].min(), "to", train['Date'].max())
print("Test date range :", test['Date'].min(),  "to", test['Date'].max())

Train date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
Test date range : 2015-08-01 00:00:00 to 2015-09-17 00:00:00


In [14]:
train.to_csv('data/train_clean.csv', index=False)
test.to_csv('data/test_clean.csv', index=False)

print("Saved! train_clean.csv and test_clean.csv are in your data folder")

Saved! train_clean.csv and test_clean.csv are in your data folder
